In [1]:
import pandas as pd

In [7]:
topic_category = {
    '文化与旅游': 'Culture',
    '医疗与公共健康': 'Health',
    '政治与政策': 'Politics',
    '民生与便民服务': 'Livelihood',
    '经济与金融': 'Economy',
    '就业与培训': 'Employment',
    '交通与基础设施': 'Transport',
    '应急与安全': 'Safety',
    '教育与人才': 'Edu & Talent',
    '法律与治安管理': 'Law',
    '环境与可持续发展': 'Environment',
    '体育赛事': 'Sports',
    '科技与创新': 'Science',
    '其他': 'Others'
}

In [8]:
local = pd.read_csv('./data/topic-results/result_local_topic.csv')[['province', 'city_name', 'old_like_num', 'read_num', 'share_num', 'label']]
non = pd.read_csv('./data/topic-results/result_non_local.csv')[['province', 'city_name', 'old_like_num', 'read_num', 'share_num', 'label']]

In [9]:
local

,province,city_name,old_like_num,read_num,share_num,label
0,澳门,澳门特别行政区,15,5052,12,体育赛事
1,澳门,澳门特别行政区,15,3523,0,文化与旅游@应急与安全
2,澳门,澳门特别行政区,6,1980,25,教育与人才@就业与培训
3,澳门,澳门特别行政区,9,2492,0,文化与旅游@应急与安全
4,澳门,澳门特别行政区,12,1569,0,文化与旅游@交通与基础设施@应急与安全
...,...,...,...,...,...,...
2063,澳门,澳门特别行政区,14,1769,0,政治与政策@经济与金融@科技与创新
2064,澳门,澳门特别行政区,3,2795,0,医疗与公共健康
2065,澳门,澳门特别行政区,5,1901,0,医疗与公共健康
2066,澳门,澳门特别行政区,14,14537,0,医疗与公共健康


In [27]:
def calc_category_mean(df, category_map=None, drop_others=True):
    df = df.copy()
    # 拆分多标签（按 @ 分割）
    df['label'] = df['label'].str.split('@')
    df = df.explode('label')
    # 去除空值
    df = df.dropna(subset=['label'])
    # 分组计算平均值
    result = (
        df.groupby('label')[['old_like_num', 'read_num', 'share_num']]
        .mean()
        .reset_index()
    )
    # 如果提供英文映射
    if category_map is not None:
        result['label_en'] = result['label'].map(category_map)
        if drop_others:
            result = result[result['label_en'] != 'Others']
            
    return result.sort_values('read_num', ascending=False)

In [28]:
local_reads = calc_category_mean(local, topic_category)
local_reads.to_excel('./analysis-data/puclic_engagement_local_reads.xlsx', index=None)
local_reads

,label,old_like_num,read_num,share_num,label_en
3,医疗与公共健康,10.543119,4617.957798,0.216514,Health
5,应急与安全,7.471545,1949.804878,0.130081,Safety
0,交通与基础设施,9.685990,1894.608696,0.995169,Transport
7,教育与人才,9.305195,1848.000000,1.649351,Edu & Talent
6,政治与政策,11.256494,1845.305195,0.756494,Politics
1,体育赛事,12.042857,1779.671429,1.685714,Sports
10,法律与治安管理,9.770492,1710.950820,0.237705,Law
12,科技与创新,10.482759,1687.747126,0.942529,Science
8,文化与旅游,9.279699,1565.542857,1.151880,Culture
13,经济与金融,8.487352,1492.109612,0.534570,Economy


In [29]:
non_reads = calc_category_mean(non, topic_category)
non_reads.to_excel('./analysis-data/puclic_engagement_non_reads.xlsx', index=None)
non_reads

,label,old_like_num,read_num,share_num,label_en
0,交通与基础设施,37.426282,18227.056090,3.336538,Transport
5,应急与安全,31.973684,17932.631579,0.473684,Safety
9,民生与便民服务,26.487896,16225.774674,8.538175,Livelihood
1,体育赛事,41.938931,15221.229008,1.595420,Sports
3,医疗与公共健康,24.503597,14876.258993,3.201439,Health
4,就业与培训,48.085271,13807.651163,5.410853,Employment
12,科技与创新,28.447552,12997.755245,3.629371,Science
8,文化与旅游,41.743202,12863.613293,5.861027,Culture
10,法律与治安管理,19.464789,12127.746479,3.366197,Law
13,经济与金融,35.996681,11105.318584,3.848451,Economy


In [30]:
local_renamed = local_reads.rename(
    columns={c: f'LOCAL_{c}' for c in local_reads.columns if c not in ['label', 'label_en']})

non_renamed = non_reads.rename(
    columns={c: f'NON_LOCAL_{c}' for c in non_reads.columns if c not in ['label', 'label_en']})

merged_df = local_renamed.merge(
    non_renamed,
    on=['label', 'label_en'],
    how='outer'   
)

cols = [
    'label', 'label_en',
    'LOCAL_old_like_num', 'LOCAL_read_num', 'LOCAL_share_num',
    'NON_LOCAL_old_like_num', 'NON_LOCAL_read_num', 'NON_LOCAL_share_num'
]


merged_df = merged_df[cols]

num_cols = merged_df.select_dtypes(include='number').columns
merged_df[num_cols] = merged_df[num_cols].apply(
    lambda col: col.map(lambda x: f'{x:.2f}' if pd.notnull(x) else ''))

merged_df.to_excel('./analysis-data/local_nonlocal_merged.xlsx', index=False)
merged_df

,label,label_en,LOCAL_old_like_num,LOCAL_read_num,LOCAL_share_num,NON_LOCAL_old_like_num,NON_LOCAL_read_num,NON_LOCAL_share_num
0,交通与基础设施,Transport,9.69,1894.61,1.00,37.43,18227.06,3.34
1,体育赛事,Sports,12.04,1779.67,1.69,41.94,15221.23,1.60
2,医疗与公共健康,Health,10.54,4617.96,0.22,24.50,14876.26,3.20
3,就业与培训,Employment,4.21,802.57,0.55,48.09,13807.65,5.41
4,应急与安全,Safety,7.47,1949.80,0.13,31.97,17932.63,0.47
5,政治与政策,Politics,11.26,1845.31,0.76,21.90,5323.47,1.03
6,教育与人才,Edu & Talent,9.31,1848.00,1.65,22.62,10134.48,4.62
7,文化与旅游,Culture,9.28,1565.54,1.15,41.74,12863.61,5.86
8,民生与便民服务,Livelihood,7.25,1456.37,0.88,26.49,16225.77,8.54
9,法律与治安管理,Law,9.77,1710.95,0.24,19.46,12127.75,3.37


In [83]:
def get_rank(cl_name):
    temp = merged_df[['label', 'label_en', cl_name]].copy()
    temp[cl_name] = pd.to_numeric(temp[cl_name])
    temp = temp.sort_values(by=cl_name, ascending=False).reset_index(drop=True)
    temp[cl_name + '_RANK'] = temp.index + 1
    return temp[['label_en', cl_name + '_RANK']]
    
get_rank('LOCAL_read_num')    

,label_en,LOCAL_read_num_RANK
0,Health,1
1,Safety,2
2,Transport,3
3,Edu & Talent,4
4,Politics,5
5,Sports,6
6,Law,7
7,Science,8
8,Culture,9
9,Economy,10


In [84]:
get_rank('LOCAL_old_like_num')    

,label_en,LOCAL_old_like_num_RANK
0,Sports,1
1,Politics,2
2,Health,3
3,Science,4
4,Law,5
5,Transport,6
6,Edu & Talent,7
7,Culture,8
8,Economy,9
9,Safety,10


In [85]:
get_rank('LOCAL_share_num')    

,label_en,LOCAL_share_num_RANK
0,Sports,1
1,Edu & Talent,2
2,Culture,3
3,Transport,4
4,Science,5
5,Livelihood,6
6,Environment,7
7,Politics,8
8,Employment,9
9,Economy,10


In [117]:
def show_rank():
    local_R1 = get_rank('LOCAL_read_num')  
    local_R2 = get_rank('LOCAL_old_like_num') 
    local_R3 = get_rank('LOCAL_share_num')
    local_R = local_R1.merge(local_R2, on='label_en').merge(local_R3, on='label_en')
    local_R['LOCAL_sum_RANK'] = local_R[['LOCAL_read_num_RANK', 'LOCAL_old_like_num_RANK', 'LOCAL_share_num_RANK']].sum(axis=1)
    # 按总排名升序排序
    local_R = local_R.sort_values(
        by=['LOCAL_sum_RANK', 'LOCAL_read_num_RANK'],
        ascending=[True, True]
    ).reset_index(drop=True)

    local_R['engagement_RANK'] = range(1, len(local_R) + 1)
    
    
    topic_fre_rank = pd.read_excel('./analysis-data/topic_rank_diff.xlsx')
    topic_fre_rank['frequency_RANK'] = topic_fre_rank['RANK']
    topic_fre_rank = topic_fre_rank[topic_fre_rank.source=='Local']
    topic_fre_rank['label_en'] = topic_fre_rank['topic_en']

    engagement_rank_df = local_R.copy()
    frequency_rank_df = topic_fre_rank[['label_en', 'frequency_RANK']].copy()
    final_df = pd.merge(engagement_rank_df, frequency_rank_df, on='label_en')
    
    final_df['diff'] = final_df['engagement_RANK'] - final_df['frequency_RANK']
    final_df.to_excel('./analysis-data/RANK_vs_RANK.xlsx', index=None)
    return final_df

In [118]:
RRR = show_rank()
RRR

,label_en,LOCAL_read_num_RANK,LOCAL_old_like_num_RANK,LOCAL_share_num_RANK,LOCAL_sum_RANK,engagement_RANK,frequency_RANK,diff
0,Sports,6,1,1,8,1,13,-12
1,Transport,3,6,4,13,2,6,-4
2,Edu & Talent,4,7,2,13,3,8,-5
3,Politics,5,2,8,15,4,5,-1
4,Health,1,3,12,16,5,3,2
5,Science,8,4,5,17,6,12,-6
6,Culture,9,8,3,20,7,1,6
7,Law,7,5,11,23,8,10,-2
8,Safety,2,10,13,25,9,9,0
9,Economy,10,9,10,29,10,2,8


In [119]:
RRR['diff'].sum()

np.int64(0)